# 00 — Setup: Catalog, Schema, and Volume (Ayurveda chatbot / AyurGenix)

**What this notebook does**
1. Creates catalog `ayurveda_lakehouse` for your **Ayurveda / AyurGenix** lakehouse (Unity Catalog + managed volume).
2. Creates schema `ayurgenix`.
3. Creates UC Volume `source_vault` for `AyurGenixAI_Dataset.csv` and your **PDF** sources for RAG.
4. Copies every **file** from repo `raw_data/` into the Volume (no manual upload).

Later notebooks **ingest the CSV into Delta** and **extract text from all PDFs** into chunk tables for the chatbot.

**Put in `raw_data/` before you run**
- `AyurGenixAI_Dataset.csv` — from [AyurGenixAI on Kaggle](https://www.kaggle.com/datasets/kagglekirti123/ayurgenixai-ayurvedic-dataset)
- `ayurveda.pdf`, `ayurveda_offering_herbal_healing.pdf`, `ayurveda_the_science_of_life_dossier.pdf`

**How to run** — attach **Serverless** (or UC cluster), then **Run all**.

> If `CREATE CATALOG` fails on permissions, use catalog `workspace` and replace `ayurveda_lakehouse.ayurgenix` / volume paths with `workspace.ayurgenix` / `/Volumes/workspace/...` in later notebooks.


## Step 1 — Create the catalog


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ayurveda_lakehouse
COMMENT 'Ayurveda chatbot lakehouse (AyurGenix + PDF RAG sources).';

USE CATALOG ayurveda_lakehouse;


## Step 2 — Create the schema (database)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ayurgenix
COMMENT 'Bronze/silver objects for structured AyurGenix data and document paths.';

USE SCHEMA ayurgenix;


## Step 3 — Create a Volume for raw files
Non-tabular landing: CSV + PDFs.


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ayurveda_lakehouse.ayurgenix.source_vault
COMMENT 'AyurGenix CSV and ayurveda PDFs for ingestion and RAG prep';


## Step 4 — Copy files from `raw_data/` into the Volume
Discover the repo root (two levels up from this notebook), then copy each file from `raw_data/` into the Volume.


In [0]:
import os, shutil

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
repo_root = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))
raw_data_dir = os.path.join(repo_root, "raw_data")
volume_path = "/Volumes/ayurveda_lakehouse/ayurgenix/source_vault"

print(f"Repo root: {repo_root}")
print(f"Raw data dir: {raw_data_dir}")
print(f"Volume target: {volume_path}")
print()

copied = 0
for name in sorted(os.listdir(raw_data_dir)):
    src = os.path.join(raw_data_dir, name)
    if not os.path.isfile(src):
        continue
    dst = os.path.join(volume_path, name)
    shutil.copyfile(src, dst)
    size = os.path.getsize(dst)
    print(f" copied {name} ({size:,} bytes)")
    copied += 1

print(f"\nDone. {copied} files copied into the Volume.")


## Step 5 — Verify files in the Volume


In [0]:
files = dbutils.fs.ls("/Volumes/ayurveda_lakehouse/ayurgenix/source_vault")
for f in files:
    print(f.name, "-", f.size, "bytes")


**Next step:** open `01_ingest_ayurgenix_dataset`.
